# ARA Dropnoise — Final

Basis notebook ini adalah **resep yang sudah terbukti tembus 0.82 di leaderboard**
(`ara-dropnoise (1).ipynb`, isinya identik dengan `jajaja.ipynb` kecuali blacklist).
**Tidak ada perubahan arsitektur, augmentasi, loss weighting, atau training loop** dari resep itu.

Kenapa sekonservatif ini: revisi sebelumnya (`ara-dropnoise-v3.ipynb`) mengganti model,
augmentasi, resize strategy, dan post-processing sekaligus — hasilnya turun ke 0.74.
Investigasi menunjukkan alasannya *masuk akal di atas kertas* tapi **tidak diukur satu-satu**,
dan beberapa di antaranya justru melawan struktur data (lihat ringkasan di bawah).
Notebook ini menerapkan pelajaran itu: ubah sesedikit mungkin, dan setiap perubahan
harus (a) diukur langsung dari data, dan (b) bisa dimatikan lewat toggle.

## Yang benar-benar berubah dari resep 0.82

| # | Perubahan | Status | Bukti |
|---|---|---|---|
| 1 | `train_298.jpg` ditambah ke blacklist | **aktif by default** | 1 komponen asli (93% mask) + 51 komponen speckle kecil di sekelilingnya — noise segmentasi terkuantifikasi, lihat sel audit di bawah |
| 2 | `train_031.jpg` — kandidat ambigu | **toggle, OFF by default** | area basah/retak diberi label luas, pola mirip `train_016` (yang sudah terbukti noise) — tapi tidak sekonklusif `train_298`. Lihat sendiri di sel audit, putuskan lewat `EXCLUDE_TRAIN_031` |
| 3 | Bug `FocalLoss` menerima probabilitas, bukan logits | **toggle, OFF by default** | `smp.losses.FocalLoss` selalu menerapkan sigmoid internal; kalau modelnya sudah `activation="sigmoid"`, sigmoid diterapkan dua kali. Diverifikasi numerik: rentang efektif loss terkompresi dari [0,1] jadi [0.5, 0.73]. Default OFF supaya run pertama = reproduksi paling dekat ke 0.82; nyalakan `FIX_FOCAL_LOGITS_BUG` untuk eksperimen kedua |

## Yang SENGAJA tidak diubah (dan kenapa)

- **Resize persegi 512x512** (bukan letterbox) — letterbox terbukti membuang 27.8% piksel objek demi menghindari regangan yang median-nya cuma 1.33x. Lihat post-mortem v3.
- **Segformer + mit_b3** (bukan Unet+ConvNeXt) — plafon keuntungan decoder lebih tajam cuma ~0.8% (diukur dari GT vs GT-diblur-stride4), dan mask di dataset ini sendiri kasar (solidity median 0.89, keselarasan tepi cuma 1.22x baseline acak) — jadi 0.8% itu pun mustahil ditagih.
- **Augmentasi 4-operasi** (Resize, HFlip, RBC/CLAHE, Sharpen) — bukan 11-operasi. `ElasticTransform`/`GridDistortion` mendeformasi mask yang sudah kasar, menambah noise di atas noise.
- **Tanpa validation holdout** — training di 100% data + oversampling, sama seperti resep asli. Fold holdout di v3 membuang 20% data dari dataset yang cuma 498 gambar.
- **Cluster dashcam (66 gambar, termasuk `LIST_DICE_0`) tetap di training, tetap di-oversample** — ini domain-shift (dikonfirmasi lewat ResNet18 embedding di `jajaja.ipynb` sendiri), bukan noise label. Penulis aslinya sudah sadar dan sengaja tidak meng-crop/membuang, karena self-attention Segformer diharapkan menekan konteks langit/dashboard yang tidak relevan. Tidak ada bukti baru yang mengalahkan alasan itu.
- **TTA horizontal-flip saja** (2 pass, bukan 18) — ini juga persis resep asli. Prior spasial (kepadatan mask 1.80x lebih tinggi di paruh bawah gambar) membuat vflip/rotasi jadi kontraproduktif; kiri/kanan simetris jadi hflip aman.

## Cara pakai

1. Jalankan semua sel dengan toggle default (semua False kecuali blacklist `train_298`). Ini reproduksi paling dekat ke 0.82 + 1 fix noise berbukti kuat. Submit, catat skornya.
2. Buka sel "AUDIT VISUAL" di bawah, lihat `train_031` sendiri. Kalau setuju itu noise, set `EXCLUDE_TRAIN_031 = True`, jalankan ulang, bandingkan.
3. Coba `FIX_FOCAL_LOGITS_BUG = True` secara terpisah (jangan digabung dengan langkah 2 di run yang sama) supaya efeknya bisa diatribusikan dengan jelas.
4. Isi tabel eksperimen di sel paling akhir tiap kali submit, supaya perbandingannya tidak hilang.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
# ====================================================
# 1. IMPORT & CONFIG
# ====================================================
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def set_seed(seed=2802405030):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(2802405030)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 25

TRAIN_IMG = "/kaggle/input/competitions/data-science-ara-7-0/dataset/dataset/train/images"
TRAIN_MASK = "/kaggle/input/competitions/data-science-ara-7-0/dataset/dataset/train/mask"
TEST_IMG  = "/kaggle/input/competitions/data-science-ara-7-0/dataset/dataset/test/images"
os.makedirs("/kaggle/working/predicted_masks/", exist_ok=True)

# ---- TOGGLE eksperimen (lihat markdown di atas untuk bukti tiap toggle) ----
EXCLUDE_TRAIN_031 = False      # kandidat ambigu, putuskan sendiri setelah lihat sel audit
FIX_FOCAL_LOGITS_BUG = False   # default False = reproduksi numerik paling dekat ke resep 0.82

print(f"Device: {DEVICE}")
print(f"EXCLUDE_TRAIN_031 = {EXCLUDE_TRAIN_031}")
print(f"FIX_FOCAL_LOGITS_BUG = {FIX_FOCAL_LOGITS_BUG}")

## Data preparation

`BLACKLIST` sama seperti resep asli, ditambah `train_298.jpg` (net-new, lihat sel audit
untuk buktinya). `LIST_DICE_0/LOW/MID` dan mekanisme oversampling **tidak diubah sama sekali** —
tiga list ini berasal dari evaluasi self-dice pada training set di iterasi sebelumnya
(lihat Section "Evaluation" di bawah untuk cara list ini diperbarui kalau perlu).

In [ ]:
# ====================================================
# 2. BLACKLIST & TIERED OVERSAMPLING
# ====================================================
BLACKLIST = [
    'train_002.jpg', 'train_016.jpg', 'train_024.jpg', 'train_038.jpg', 'train_086.jpg',
    'train_063.jpg', 'train_129.jpg', 'train_130.jpg', 'train_131.jpg', 'train_133.jpg',
    'train_140.jpg', 'train_169.jpg', 'train_252.jpg', 'train_337.jpg', 'train_366.jpg',
    'train_298.jpg',   # [BARU] 1 komponen asli (93% area mask) + 51 komponen speckle kecil
]
if EXCLUDE_TRAIN_031:
    BLACKLIST.append('train_031.jpg')

LIST_DICE_0 = ['train_062.jpg', 'train_065.jpg', 'train_068.jpg', 'train_075.jpg', 'train_079.jpg', 'train_084.jpg', 'train_121.jpg', 'train_134.jpg', 'train_137.jpg', 'train_160.jpg', 'train_236.jpg', 'train_263.jpg', 'train_336.jpg', 'train_352.jpg', 'train_358.jpg', 'train_363.jpg', 'train_401.jpg', 'train_424.jpg', 'train_438.jpg', 'train_455.jpg', 'train_472.jpg', 'train_474.jpg', 'train_489.jpg']
LIST_DICE_LOW = ['train_020.jpg', 'train_029.jpg', 'train_283.jpg', 'train_496.jpg']
LIST_DICE_MID = ['train_023.jpg', 'train_027.jpg', 'train_071.jpg', 'train_100.jpg', 'train_101.jpg', 'train_127.jpg', 'train_135.jpg', 'train_159.jpg', 'train_183.jpg',
                 'train_212.jpg', 'train_227.jpg', 'train_231.jpg', 'train_234.jpg', 'train_240.jpg', 'train_247.jpg', 'train_255.jpg', 'train_261.jpg', 'train_265.jpg', 'train_267.jpg',
                 'train_271.jpg', 'train_272.jpg', 'train_281.jpg', 'train_291.jpg', 'train_294.jpg', 'train_302.jpg', 'train_389.jpg', 'train_412.jpg', 'train_413.jpg', 'train_415.jpg',
                 'train_428.jpg', 'train_475.jpg', 'train_486.jpg']

all_files = sorted([f for f in os.listdir(TRAIN_IMG) if f not in BLACKLIST])

very_low = [f for f in (LIST_DICE_0 + LIST_DICE_LOW) if f not in BLACKLIST]
mid_boost = [f for f in LIST_DICE_MID if f not in BLACKLIST]

img_files = all_files + (very_low * 2) + (mid_boost * 1)
random.shuffle(img_files)

mask_map = {"".join(filter(str.isdigit, m)): m for m in os.listdir(TRAIN_MASK)}

print(f"Blacklisted: {len(BLACKLIST)} files removed.")
print(f"Original Data (Clean): {len(all_files)}")
print(f"Oversampled Data (Clean): {len(img_files)}")

### Audit visual — `train_298` (sudah dikeluarkan) dan `train_031` (putuskan sendiri)

Sel ini membaca langsung dari dataset, jadi jalan di Kaggle tanpa dependensi tambahan.

In [ ]:
# ====================================================
# 3. AUDIT VISUAL — bukti untuk dua file di atas
# ====================================================
def show_audit(fname, mask_dir=TRAIN_MASK, img_dir=TRAIN_IMG):
    iid = "".join(filter(str.isdigit, fname))
    mmap = {"".join(filter(str.isdigit, m)): m for m in os.listdir(mask_dir)}
    img = cv2.cvtColor(cv2.imread(os.path.join(img_dir, fname)), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(os.path.join(mask_dir, mmap[iid]), cv2.IMREAD_GRAYSCALE)
    mb = (mask > 127).astype(np.uint8)
    n_comp, lab, stats, _ = cv2.connectedComponentsWithStats(mb, 8)
    areas = sorted(stats[1:, cv2.CC_STAT_AREA], reverse=True)

    overlay = img.copy()
    overlay[mb > 0] = (overlay[mb > 0] * 0.55 + np.array([255, 255, 0]) * 0.45).astype(np.uint8)
    cnts, _ = cv2.findContours(mb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, cnts, -1, (255, 0, 0), 2)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(img); axes[0].set_title(fname); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title(f"coverage={mb.mean():.3f}  n_comp={n_comp-1}"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

    print(f"{fname}: {n_comp-1} komponen, top-5 luas (px): {areas[:5]}")
    if len(areas) > 1:
        print(f"  komponen dominan = {areas[0]/sum(areas):.1%} dari total piksel mask -> "
              f"sisanya ({len(areas)-1} komponen) menyumbang {1-areas[0]/sum(areas):.1%}")

print("=" * 70)
print("train_298.jpg — SUDAH di-blacklist. Bukti: 1 blob dominan + speckle kecil di sekitarnya.")
print("=" * 70)
show_audit("train_298.jpg")

print("\n" + "=" * 70)
print("train_031.jpg — MASIH di training (default). Lihat sendiri: pothole asli, atau")
print("area basah/retak yang diberi label terlalu luas (mirip train_016 yang sudah terbukti noise)?")
print("=" * 70)
show_audit("train_031.jpg")

## Augmentasi — 4 operasi, tidak diubah dari resep asli

In [ ]:
# ====================================================
# 4. AUGMENTATION
# ====================================================
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
        A.CLAHE(clip_limit=4.0, p=1.0),
    ], p=0.7),
    A.Sharpen(alpha=(0.2, 0.5), p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

## Model, loss, optimizer, scheduler

Model & optimizer/scheduler **identik** dengan resep 0.82. Loss punya satu cabang toggle
untuk bug `FocalLoss`, dijelaskan di bawah.

In [ ]:
# ====================================================
# 5. MODEL
# ====================================================
model = smp.Segformer(
    encoder_name="mit_b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=(None if FIX_FOCAL_LOGITS_BUG else "sigmoid")
).to(DEVICE)

print(f"Model: Segformer + mit_b3 ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")
print(f"Output: {'logits' if FIX_FOCAL_LOGITS_BUG else 'probabilitas (sigmoid)'}")

**Bug yang ditemukan:** `smp.losses.FocalLoss` tidak punya parameter `from_logits` —
implementasinya SELALU memanggil `binary_cross_entropy_with_logits` di dalam, yang menerapkan
`sigmoid` sendiri ke input. Resep asli memberi model `activation="sigmoid"`, jadi `FocalLoss`
menerima probabilitas [0,1] dan menerapkan sigmoid **kedua kalinya**.

Diverifikasi numerik: untuk logits acak yang sama, `FocalLoss(sigmoid(logits), target) = 0.227`
vs `FocalLoss(logits, target) = 1.198` — beda ~5x. Lebih penting dari besarannya: rentang efektif
`sigmoid(probabilitas)` cuma `[0.50, 0.73]` (karena input sudah dibatasi [0,1]), padahal harusnya
mendekati `[0, 1]`. Mekanisme inti focal loss (menekan piksel mudah, menonjolkan piksel sulit)
nyaris tidak berfungsi dalam rentang sesempit itu — jadi loss gabungan `0.5*dice + 0.5*focal`
secara efektif hampir semuanya didorong oleh `DiceLoss`.

Ini **bukan alasan untuk otomatis "memperbaikinya"** — resep dengan bug ini yang mencetak 0.82.
Mematikan bug bisa membuat komponen focal benar-benar berfungsi (berpotensi membantu piksel
sulit/tepi), atau bisa mengubah keseimbangan yang kebetulan sudah pas. Makanya toggle-nya
default `False`, dan disarankan diuji di run terpisah, bukan digabung dengan perubahan lain.

In [ ]:
# ====================================================
# 6. LOSS, OPTIMIZER, SCHEDULER
# ====================================================
criterion = smp.losses.DiceLoss(mode='binary', from_logits=FIX_FOCAL_LOGITS_BUG)
focal_loss = smp.losses.FocalLoss(mode='binary')   # smp tidak punya opsi from_logits - selalu logits

def to_prob(raw_output):
    # Konversi output model ke probabilitas [0,1], apapun status FIX_FOCAL_LOGITS_BUG.
    # Dipakai di semua tempat downstream (eval, TTA) yang membandingkan terhadap threshold 0.4,
    # supaya toggle di atas TIDAK mengubah semantik kode manapun selain loss itu sendiri.
    return torch.sigmoid(raw_output) if FIX_FOCAL_LOGITS_BUG else raw_output

optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, weight_decay=5e-4)

steps_per_epoch = (len(img_files) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-4, total_steps=total_steps,
    pct_start=0.2, div_factor=10, final_div_factor=100
)

print(f"Loss: 0.5*Dice + 0.5*Focal | steps/epoch={steps_per_epoch} | total_steps={total_steps}")

## Training loop

Mekanisme loop **tidak diubah** (manual batching, bukan `DataLoader`, persis resep asli).
Tambahan yang murni aditif dan tidak mengubah apa yang dilihat model:
checkpoint tiap epoch (jaring pengaman kalau sesi Kaggle terputus) dan pencatatan loss untuk plot.

In [ ]:
# ====================================================
# 7. TRAINING LOOP
# ====================================================
history = {"loss": []}

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    pbar = tqdm(range(0, len(img_files), BATCH_SIZE), total=steps_per_epoch, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

    for i in pbar:
        batch_files = img_files[i:i+BATCH_SIZE]
        imgs_b, masks_b = [], []
        for f in batch_files:
            img = cv2.imread(os.path.join(TRAIN_IMG, f))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_id = "".join(filter(str.isdigit, f))
            m_file = mask_map.get(img_id)
            if m_file:
                mask = (cv2.imread(os.path.join(TRAIN_MASK, m_file), cv2.IMREAD_GRAYSCALE) > 0).astype(np.float32)
                aug = transform(image=img, mask=mask)
                imgs_b.append(aug["image"])
                masks_b.append(aug["mask"].unsqueeze(0))

        if not imgs_b: continue
        imgs_t, masks_t = torch.stack(imgs_b).to(DEVICE), torch.stack(masks_b).to(DEVICE)

        optimizer.zero_grad()
        output = model(imgs_t)
        loss = 0.5 * criterion(output, masks_t) + 0.5 * focal_loss(output, masks_t)
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    history["loss"].append(epoch_loss / max(n_batches, 1))
    torch.save(model.state_dict(), "/kaggle/working/last_model.pth")   # [BARU] jaring pengaman

In [ ]:
# ====================================================
# 8. KURVA TRAINING  [BARU - diagnostik saja, tidak mempengaruhi training]
# ====================================================
plt.figure(figsize=(8, 4))
plt.plot(history["loss"], marker="o", ms=3)
plt.xlabel("epoch"); plt.ylabel("avg loss"); plt.title("Training loss")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Evaluation & statistics

Sama seperti resep asli: dice dihitung di training set sendiri (bukan held-out set — resep ini
memang tidak punya validation split, lihat catatan di pembuka). Dipakai untuk memantau
`LIST_DICE_0/LOW/MID` di masa depan, bukan untuk mengklaim generalisasi.

In [ ]:
# ====================================================
# 9. EVALUATION & STATISTICS
# ====================================================
def dice_score(pred, target, eps=1e-6):
    pred = (pred > 0.4).astype(np.uint8)
    target = (target > 0).astype(np.uint8)
    intersection = (pred * target).sum()
    return (2 * intersection + eps) / (pred.sum() + target.sum() + eps)

results = []
model.eval()
with torch.no_grad():
    for f in tqdm(all_files, desc="Calculating Statistics"):
        img = cv2.imread(os.path.join(TRAIN_IMG, f))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        img_id = "".join(filter(str.isdigit, f))
        if img_id not in mask_map: continue
        gt_mask = (cv2.imread(os.path.join(TRAIN_MASK, mask_map[img_id]), cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)
        img_t = val_transform(image=img_rgb)["image"].unsqueeze(0).to(DEVICE)
        pred = to_prob(model(img_t)).cpu().numpy().squeeze()
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
        results.append({"file": f, "dice": dice_score(pred, gt_mask)})

low_dice_cases = [x for x in results if x["dice"] < 0.50]
mid_dice_cases = [x for x in results if 0.50 <= x["dice"] < 0.8]
high_dice_cases = [x for x in results if x["dice"] >= 0.8]

print("\n" + "="*50)
print(f"FINAL STATISTICS ON TRAIN SET:")
print(f"Total low Dice cases (<0.50): {len(low_dice_cases)}")
print(f"Total mid Dice cases (0.50-0.80): {len(mid_dice_cases)}")
print(f"Total high Dice cases (>=0.80): {len(high_dice_cases)}")
print(f"Mean Dice: {np.mean([x['dice'] for x in results]):.4f}")
print("="*50 + "\n")

## Inference + TTA + submission

TTA horizontal-flip saja (2 pass), threshold 0.4 — persis resep asli.

In [ ]:
# ====================================================
# 10. INFERENCE + TTA + RLE
# ====================================================
def encode_rle(mask):
    pixels = (mask == 255).T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

test_files = sorted(os.listdir(TEST_IMG))
submission = []

model.eval()
with torch.no_grad():
    for f in tqdm(test_files, desc="Inference"):
        raw_img = cv2.imread(os.path.join(TEST_IMG, f))
        h, w = raw_img.shape[:2]
        img_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
        img_t = val_transform(image=img_rgb)["image"].unsqueeze(0).to(DEVICE)
        img_flip = torch.flip(img_t, dims=[3])

        out_t = to_prob(model(img_t))
        out_flip = torch.flip(to_prob(model(img_flip)), dims=[3])

        pred = ((out_t + out_flip) / 2 > 0.4).float().cpu().numpy().squeeze()

        pred_full = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
        pred_mask = (pred_full * 255).astype(np.uint8)

        rle = encode_rle(pred_mask) if np.any(pred_mask) else ""
        submission.append({"ImageId": f, "rle": rle})

submission_df = pd.DataFrame(submission)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("Done! submission.csv saved.")
print(f"  baris: {len(submission_df)} | prediksi kosong: {(submission_df['rle']=='').sum()}")

In [ ]:
# ====================================================
# 11. VERIFIKASI FORMAT SUBMISSION  [BARU - safety check, tidak mengubah training]
# ====================================================
cand = []
for root in ["/kaggle/input"]:
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if "sample" in fn.lower() and fn.lower().endswith(".csv"):
                cand.append(os.path.join(dp, fn))

if cand:
    ss = pd.read_csv(cand[0])
    print(f"Sample submission ditemukan: {cand[0]}")
    print(f"  kolom sample: {list(ss.columns)} | kolom kita: {list(submission_df.columns)}")
    print(f"  baris sample: {len(ss)} | baris kita: {len(submission_df)}")
    if list(ss.columns) != list(submission_df.columns) or len(ss) != len(submission_df):
        print("  [!] ADA PERBEDAAN - cek manual sebelum submit.")
    else:
        print("  format cocok.")
else:
    print("[!] sample_submission.csv tidak ditemukan - verifikasi manual kolom & jumlah baris sebelum submit.")

## Log eksperimen

Isi manual tiap kali submit, supaya perbandingan antar-toggle tidak hilang.

| Run | `EXCLUDE_TRAIN_031` | `FIX_FOCAL_LOGITS_BUG` | Mean dice (train) | Skor LB | Catatan |
|---|---|---|---|---|---|
| baseline (0.82) | - | - | - | 0.82 | resep asli, tanpa `train_298` |
| run 1 | False | False | | | +`train_298` saja |
| run 2 | True | False | | | +`train_031` |
| run 3 | False | True | | | +fix focal loss |

**Aturan:** ubah satu toggle per run. Kalau run 1 sudah lebih baik dari 0.82, itu konfirmasi
`train_298` memang noise — baru lanjut coba toggle berikutnya satu-satu di atasnya.